# 1. Reactant DFT calculations

Input: selected 4π and 2π SMILES CSV. Output: optimized reactants, single-point energies, and `all_smiles.csv` in `SMILES_DIR`. The optional previous reactant table is configured explicitly.

Saved outputs were cleared in this split. The archived source notebook retains the historical outputs and execution counts. Set the job paths before running any calculation.


In [ ]:
from pathlib import Path
import os
import sys

# Resolve the repository root from the kernel working directory, or set CYCLOADD_REPO_ROOT.
repo_root_env = os.environ.get("CYCLOADD_REPO_ROOT")
if repo_root_env:
    REPO_ROOT = Path(repo_root_env).expanduser().resolve()
    if not (REPO_ROOT / "Code").is_dir() or not (REPO_ROOT / "Data").is_dir():
        raise RuntimeError(f"CYCLOADD_REPO_ROOT is not a repository root: {REPO_ROOT}")
else:
    REPO_ROOT = next(
        (parent for parent in (Path.cwd(), *Path.cwd().parents)
         if (parent / "Code").is_dir() and (parent / "Data").is_dir()),
        None,
    )
if REPO_ROOT is None:
    raise RuntimeError("Start Jupyter in this repository or set CYCLOADD_REPO_ROOT")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import pandas as pd
from rdkit import Chem
from Code import main, xtb_process

SMILES_DIR = os.environ.get("CYCLOADD_SMILES_DIR", r"E:\work\Secondary_Selection\Diene_Ene_Smiles")
SELECTED_SMILES_CSV = REPO_ROOT / "Data" / "ZINC_Data" / "all_selected_smiles.csv"
OLD_SMILES_CSV = None  # Set to a previous all_smiles.csv only when extending that index.


# 1. DFT Calculate the Reactants

In [ ]:
target_dir = SMILES_DIR     
csv_dirs = [str(SELECTED_SMILES_CSV)]                   # <- The dir you saved the reactants

In [ ]:
# Stablization
for csv_dir in csv_dirs:
    df = pd.read_csv(csv_dir)
    for row_id, row in df.iterrows():
        row['Diene'] = Chem.MolToSmiles(Chem.MolFromSmiles(row['Diene']))
        row['Ene'] = Chem.MolToSmiles(Chem.MolFromSmiles(row['Ene']))
        df.loc[row_id] = row
    df.to_csv(csv_dir, index=False)

In [ ]:
main.collect_smiles_file(csv_dirs = csv_dirs, root_dir=target_dir, sugan=False, old_smiles_csv=OLD_SMILES_CSV)

In [ ]:
xtb_process.shift_to_parra(target_dir + "/first_xtb")

In [ ]:
main.smiles_DFT_calc(target_dir)

In [ ]:
main.error_improve(target_dir, "mol_dft")

In [ ]:
main.SPE_DFT_calc(target_dir, save_chk=False)

In [ ]:
main.error_improve(target_dir, "mol_dft_eng")

In [ ]:
main.SPE_DFT_calc(target_dir, save_chk=False, opt_name = "mol_dft", eng_name = "mol_dft_eng_solvent", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=water)")

In [ ]:
main.error_improve(target_dir, "mol_dft_eng_solvent")

In [ ]:
main.SPE_DFT_calc(target_dir, save_chk=False, opt_name = "mol_dft", eng_name = "mol_dft_eng_solvent2", method="b3lyp/6-311+g(d,p) em=gd3bj scrf=(smd, solvent=1,4-dioxane)")

In [ ]:
main.smiles_result_analysis(target_dir)